In [50]:
import pyxdf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [51]:
data, header = pyxdf.load_xdf(r"C:\Users\jshin\OW_closedloopLIFU\xdf_data\sub-dave_run_4\ses-1\eeg\sub-dave_run_4_ses-1_task-dave_run_4_run-001_eeg.xdf")
data

In [52]:
stream = data[2]
df= []
df = pd.DataFrame(stream['time_series'])
df = df.rename(columns={i: f"Ch{i}" for i in range(df.shape[1])})

# Add timestamp column
df['Timestamp'] = stream['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in df.columns if col != 'Timestamp']
df = df[cols]
df

In [53]:
print("Streams found:", [s['info']['name'][0] for s in data])

In [54]:
EEG_LIFU_events = data[0]

# Create DataFrame from time_series
markers = pd.DataFrame(EEG_LIFU_events['time_series'])

# Rename the first column to 'markers'
markers.rename(columns={0: 'markers'}, inplace=True)

# Add timestamp column
markers['Timestamp'] = EEG_LIFU_events['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in markers.columns if col != 'Timestamp']
markers = markers[cols]
markers

In [55]:
lifu_on = markers[markers['markers'] == 'LIFU_ON']
lifu_on = np.array(lifu_on['Timestamp'])
lifu_on

In [56]:
start = markers[markers['markers'] == 'START_EXPERIMENT_RECEIVED']
start = start.iloc[0]['Timestamp']
start

In [57]:
buffer = []
offline_z = []
medians = []
merged_df = df
mads = []
last_theta_val = None
last_median_val = None
last_mad_val = None

SONICATION_TIME = 5 #seconds i believe  
COOLDOWN_TIME = 15 #sonication time + cooldown time 
THETA_THRESHOLD_Z = 1.5    # z-score threshold
MU =   1.92
SIGMA =  9.52
MAD_THRESHOLD = 10       # for artifact rejection in baseline collection
INITIAL_CUTOFF = 25.0   # initial power threshold to exclude extreme artifacts
BUFFER_SIZE = 500
sonication_enabled = start

# Ch05 = index 4
channel = "Ch11"

for i in range(len(merged_df)):
    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(last_theta_val)
        medians.append(last_median_val)
        mads.append(last_theta_val)
        continue
    # Warm-up
    if len(buffer) <= 200:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        medians.append(np.nan)
        mads.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6
    

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        medians.append(np.nan)
        mads.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)
    last_theta_val = theta_val
    last_median_val = median
    last_mad_val = mad

    # Z-score
    offline_z.append(theta_val)
    medians.append(median)
    mads.append(mad)

merged_df["offline z-score"] = offline_z
merged_df["median"] = medians
merged_df['mad'] = mads

In [58]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["offline z-score"].iloc[i]
    now = merged_df["Timestamp"].iloc[i]

    if now> sonication_enabled and theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now

In [59]:
merged_df["LIFU"] = LIFU
merged_df

In [60]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on_df = merged_df[mask_lifu]
lifu_on_df["Timestamp"] = lifu_on_df["Timestamp"]
lifu_on_cal = lifu_on_df[lifu_on_df["Timestamp"] > start]
lifu_on_cal

In [61]:
offline = np.array(lifu_on_cal['Timestamp'])
offline

In [62]:
online = lifu_on
online

In [63]:
online - offline[:-1]